# AI News Aggregator

## Project steps
   - ✅ Pahse 1 : Collect news from multiple sources 
   - ✅ Pahse 2 : Summarize articles using AI
   - ✅ Pahse 3 : Detect and remove duplicate news
   - ✅ Pahse 4 : Categorize articles by topic
   - ✅ Pahse 5 : Use an AI agent to rank news based on relevance and importance
   - ✅ Pahse 6 : Store the processed data in an SQL database
   - ✅ Pahse 7 : Generate and send a newsletter
   - ✅ Phase 8 — FastAPI
   - ✅ Phase 9 — Celery + Redis
   - ✅ Phase 10 — Dockerize
   - ✅ Phase 11 — Vector Search
   - ✅ Phase 12 — AI Chat Over Your News
   - ✅ Phase 13 — Frontend
   - ✅ Phase 13 — Deployment


#### create project structure ( later I will add more directories)
``` python 
# 1. Create the project folder and all subdirectories
mkdir -p app/{agents,collectors,llm,models,database,services,api,workers,scheduler}

# 2. Create all the empty Python files
touch app/agents/news_agent.py \
      app/api/main.py \
      app/collectors/{youtube,rss,blogs,twitter}.py \
      app/llm/provider.py \
      app/models/article.py \
      app/database/database.py \
      app/services/{summarizer,classifier,newsletter,pipeline}.py \
      app/api/routes.py \
      app/workers/celery_worker.py \
      app/scheduler/jobs.py \
      app/{config,main}.py
      app/promts/summarizer.py

# 3. Create root files
touch   {docker-compose.yml,Dockerfile,requirements.txt,.env,README.md}


# ports
               HOST MACHINE
------------------------------------------------
Browser → localhost:8001 (FastAPI)

Docker Compose Network
------------------------------------------------
api (8000)
   │
   ├──→ postgres:5432
   ├──→ redis:6379
   │
worker (celery)
   ├──→ redis:6379
   ├──→ postgres:5432

postgres container (5432 internal)
redis container (6379 internal)

```


| Service       | Internal Port | Host Port   | Purpose          |
| ------------- | ------------- | ----------- | ---------------- |
| FastAPI       | 8000          | 8001        | REST API         |
| Celery Worker | —             | —           | background tasks |
| Redis         | 6379          | 6379 / 6380 | broker + cache   |
| PostgreSQL    | 5432          | 5432 / 5433 | database         |

-----------------------------------------

##  ✅ Phase 1 — Collect News

Build a system that can collect news from multiple sources and return it in a single standardized format.

```
        RSS Feed
                \
        YouTube -----> Collectors -----> Standard Article Object
                /
        Blogs
```

At the end of Phase 1, running the application should produce something like:

```python
        [
        {
                "title": "...",
                "content": "...",
                "url": "...",
                "published_at": "...",
                "source": "OpenAI Blog"
        },
        ...
        ]
```
### ⭐ Steps :
1. I will use these 3 sources:

| Source       | RSS                                                                              |
| ------------ | -------------------------------------------------------------------------------- |
| OpenAI       | [https://openai.com/news/rss.xml](https://openai.com/news/rss.xml)               |
| Anthropic    | [https://www.anthropic.com/news/rss.xml](https://www.anthropic.com/news/rss.xml) |
| Hugging Face | [https://huggingface.co/blog/feed.xml](https://huggingface.co/blog/feed.xml)     |

2. Normalize the sources
Different sources expose different fields.we should normalize them into one structure.
The model should contain:
- title
- summary
- url
- published_at
- source

Every collector must return this exact model, regardless of where the article came from.

3. Create Collectors
- blogs
- rss
- youtube
 
and each collector should return the same model.
```
        Fetch RSS
        ↓
        Read entries
        ↓
        Convert entries to Article objects
        ↓
        Return a list of articles
```

4. Create a Collector Interface
Every collector should expose a collect() method.

5. Orchestrator
main.py should not know how RSS works. Instead, it simply coordinates collectors:
```
        OpenAI Collector
                │
        Anthropic Collector
                │
        HF Collector
                │
                ▼
        Merge results
                ▼
        Print
``` 
This separation is important. main.py orchestrates; collectors implement source-specific logic.

### Folder Structure After Phase 1
```
        ai-news-aggregator/
        │
        ├── app/
        │   ├── collectors/
        │   │   ├── rss.py
        │   │   ├── blogs.py
        │   │   ├── twitter.py
        │   │   └── youtube.py
        │   │
        │   ├── models/
        │   │   └── article.py
        │   │
        │   ├── config.py
        │   └── main.py
        │
        ├── requirements.txt
        ├── .env
        └── README.md
```
----------------

###  ✔ test Phase 1

In [ ]:
from app.config import RSS_FEEDS
from app.collectors.rss import RSSCollector

for source_name, url in RSS_FEEDS.items():
    collector = RSSCollector(source_name, url)
    articles = collector.collect()
    print("source_name", source_name)
    print("articles", articles)
   
print(len(articles))

# for test just keep first 5 articles
articles = articles[:2]




In [ ]:
articles

##  ✅ Phase 2 : AI Summarization
where the project starts becoming an AI Engineering project rather than just a data collection project.

- ✔ Build LLM Provider (llm/provider.py)
- ✔ Add Prompt Engineering
- ✔ Build Summarizer Service (services/summarizer.py)
- ✔ Update RSS Collector


###  ✔ test Phase 2

In [ ]:
from app.collectors.rss import RSSCollector
from app.config import RSS_FEEDS
from app.services.summarizer import Summarizer

summarizer = Summarizer()

for article in articles:
    article = summarizer.summarize(article)

    print("=" * 80)
    print(article.title)
    print()
    print(article.url)
    print()
    print("article content:" , article.content)
    print()
    print("summary:" , article.summary)
    print()

In [ ]:
articles

##  ✅  Phase 3 : remove duplicates
```
Summaries
      │
      ▼
Embedding Model
      │
      ▼
Vectors
      │
      ▼
Cosine Similarity
      │
      ▼
Duplicate Detector
      │
      ▼
Unique Articles
```
on this Phase I will use :
- Sentence embeddings
- Vector representations
- Cosine similarity
- Similarity thresholds
- Building a reusable AI service
- Preparing for vector databases like Pinecone, Weaviate, Chroma, and FAISS

Project Structure

We'll add a new service.
```
app/

    services/
        summarizer.py
        deduplicator.py

    embeddings/
        encoder.py
```

###  ✔test Phase 3

In [ ]:
from app.services.deduplicator import Deduplicator
deduplicator = Deduplicator()
unique_articles = deduplicator.remove_duplicates(articles)

print(f"Collected: {len(articles)}")
articles=unique_articles
print(f"Unique: {len(unique_articles)}")
unique_articles=[]

##  ✅ Phase 4 : categorization
The next stage is AI-powered topic classification, where each unique article is assigned categories such as:
- LLMs
- Robotics
- Computer Vision
- AI Research
- Open Source
- Startups
- Healthcare AI
- Autonomous Vehicles

Step 4 Architecture
```
Articles (unique)
      │
      ▼
LLM Classifier OR Zero-shot Model
      │
      ▼
Structured Output (JSON categories)
      │
      ▼
Validated Article
```

steps :
1. Define Taxonomy (VERY IMPORTANT) (app/services/taxonomy.py)
2. Choose Classification Method 
    - Option A (Recommended): LLM-based structured classification
        - Flexible
        - High accuracy
        - Easy to iterate
    - Option B: Zero-shot transformer model
        - Faster
        - No API cost
        - Less accurate
3. Prompt Design  (app/prompts/classifier.py)

In [ ]:
from app.services.classifier import Classifier
classifier = Classifier()
classified_articles = []

for article in articles:
    
    article = classifier.classify(article)

    print("title:", article.title)
    print("summary:", article.summary)
    print("categories:", article.categories)
    print("=" * 80)



In [ ]:
articles

##  ✅ Phase 5 : AI Ranking Agent
Architecture
```
Articles
   │
   ▼
Ranking Agent (LLM + heuristics)
   │
   ▼
Score per article (0–100)
   │
   ▼
Sorted Feed
```

ranking system :
1. Technical Impact (0–25)
2. Industry Importance (0–25)
3. Recency / Breaking News (0–20)
4. AI/ML Relevance (0–20)
5. Source Credibility (0–10)

steps : 
1. Extend Article Model
2. Ranking Prompt Design (app/prompts/ranker.py)
3. Ranking Agent Service (app/agents/news_agent.py)
4. Sorting Layer
5. Integrate into Pipeline

###  ✔ test Phase 5

In [ ]:
from app.agents.news_agent import NewsRankingAgent
ranker = NewsRankingAgent()
processed = []
for article in articles:
    article = ranker.rank(article)

    print(article.title)
    print(article.score)
    print(article.score_breakdown)
    print("=" * 90)

In [ ]:
articles

##  ✅Phase 6 — Store
Without a database:

You fetch news → process → lose everything

With PostgreSQL:

You keep all articles over time
You can query:
“what was published yesterday?”
“what topics are trending this week?”

PostgreSQL table
``` SQL
CREATE TABLE IF NOT EXISTS articles (
    id SERIAL PRIMARY KEY,
    title VARCHAR(255) NOT NULL,
    content TEXT  NULL,
    url VARCHAR(255)  NULL,
    published_at TIMESTAMP  NULL,
    source VARCHAR(255)  NULL,
    summary TEXT  NULL,
    score_breakdown JSONB  NULL,
    categories JSONB  NULL
);
```



In [ ]:
from app.database.database import insert_articles


# # create table
# query = """
# CREATE TABLE IF NOT EXISTS articles (
#     id SERIAL PRIMARY KEY,
#     title VARCHAR(255) NOT NULL,
#     content TEXT  NULL,
#     url VARCHAR(255)  NULL,
#     published_at TIMESTAMP  NULL,
#     source VARCHAR(255)  NULL,
#     summary TEXT  NULL,
#     score_breakdown JSONB  NULL,
#     categories JSONB  NULL
# );
# """
# engine.execute(query)
# insert articles   
# insert in DB

insert_articles( articles)

##  ✅ Phase 7 : Generate and send a newsletter
```
Ranked Articles
      │
      ▼
Newsletter Builder
      │
      ▼
HTML / Text Formatter
      │
      ▼
Email Service (SMTP / API)
      │
      ▼
User Inbox
```

##  ✅ Phase 8 : Fast API
Goal: turn the  script into an API service.

until now we have :

python main.py

After this step I will have :
``` 
API SERVER
   ↓
/run-pipeline
   ↓
RSS → AI → Newsletter → Email
```

- run api ` uvicorn app.api.main:app --host 0.0.0.0 --port 8001` 


steps :
- create a new file called `app/services/pipeline.py`
- move the main logic from `app/main.py` to `app/services/pipeline.py`
- in `app/api/main.py` import the pipeline function and call it in the `/run-pipeline` endpoint
- test http://192.168.2.130:8001/run-pipeline


## new Architecture
```
app/
│
├── api/
│   ├── main.py
│   └── routes/
│       ├── news.py
│       ├── pipeline.py
│
├── core/
│   ├── config.py
│   └── database.py
│
├── repositories/
│   └── article_repository.py
│
├── services/
│   ├── pipeline/
│   │   ├── orchestrator.py
│   │   ├── collector.py
│   │   ├── processor.py
│   │   └── ranking.py
│   ├── summarizer.py
│   ├── classifier.py
│   ├── deduplicator.py
│   ├── newsletter.py
│   └── email_sender.py
│
├
│
├── models/
│   └── article.py
```
Architecture
- API no longer runs heavy logic
- Pipeline is isolated service layer
- DB access is fully abstracted


Maintainability
- SQL isolated in repository
- pipeline is testable independently
- services are reusable

Production readiness
- proper DB lifecycle
- modular routing


# ✅ Phase 9 — Celery + Redis
Right now my API looks like this:
```
Client
   │
   ▼
GET /run-pipeline
   │
   ▼
Collect RSS
Summarize
Deduplicate
Classify
Rank
Generate Newsletter
Send Email
   │
   ▼
Response (after 2–5 minutes)
```

This is not how production APIs work because the HTTP request stays open while the AI pipeline runs.

Instead, we'll build this:
```
             FastAPI
                │
                ▼
      POST /run-pipeline
                │
                ▼
      Queue Background Job
                │
          Redis Queue
                │
                ▼
         Celery Worker
                │
                ▼
      AI Pipeline Executes
                │
                ▼
      Save Results / Send Email
```
- Step 9.1 — What are Redis and Celery?
    - Redis
        - Redis is an in-memory database.

        - In our project it will act as a message queue.
        - Imagine this queue:
            ```
            Pipeline Job #1

            Pipeline Job #2

            Pipeline Job #3
            ```
        - FastAPI only adds jobs to the queue.
    - Celery
        - Celery is a worker.
        - It continuously watches Redis.
            ```
            Redis

            ↓

            New Job

            ↓

            Celery notices

            ↓

            Runs your AI pipeline
            ```

steps :
- 1. Configure Celery : app/workers/celery_worker.py
    - <b> This object will be imported everywhere. </b>
- 2. Create Tasks  : app/services/tasks.py
    - <b> This is where the actual work happens. </b>
    - Notice that all your business logic stays inside pipeline.py.
    - Celery should only launch work.
- 3. update FastAPI Endpoint

# ✅ Phase 10 — Dockerize
Docker thinking
- "Each service lives in its own network namespace"
- "They talk via service names" 


At the end of this phase we should be able to start everything with one command:
``` Bash
docker compose up --build
```
and have
```
                Docker Network
┌────────────────────────────────────────────┐
│                                            │
│  FastAPI API        Celery Worker          │
│        │                 │                 │
│        └──────┬──────────┘                 │
│               │                            │
│           PostgreSQL                       │
│               │                            │
│            Redis                           │
│                                            │
└────────────────────────────────────────────┘
```

- stpe1 : create Dockerfile
- Step 2 — Create docker-compose
- run 
    - docker compose down
    - docker compose build --no-cache
    - docker-compose up -d

In [1]:
from app.workers.tasks import run_pipeline_task

# Run it synchronously for debugging
result = run_pipeline_task.apply()   # or .delay() for async
print(result.get())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded successfully
Encoding 2 articles for deduplication...
Deduplication complete: 2 → 2 articles
finish classification and ranking
finish sorting
inserted articles: 2
finish pipeline
[Article(title='Inside Genebench-Pro', source='OpenAI', score=83, summary_len=0, categories=['AI Research', 'Data Science'], content_len=0), Article(title='How ChatGPT adoption has expanded', source='OpenAI', score=56, summary_len=613, categories=['LLMs', 'AI Research'], content_len=178)]
{'inserted': 2}
